# 01 — Compile a model to xmodel (host-side)

This notebook runs on the **host PC** (NVIDIA GPU, x86_64) inside the
Vitis-AI 3.5 Docker container. It:

1. Loads a trained PyTorch checkpoint
2. Strips the detect head's inline post-processing
3. Calibrates on a folder of representative images
4. Runs `vai_q_pytorch` post-training quantization
5. Compiles the quantized graph for the KV260's B4096 DPU
6. Outputs `out/<variant>/<variant>_kv260.xmodel`

## Prerequisites

This notebook **must** run inside the Vitis-AI Docker. Two ways to launch:

**A — Headless (recommended)**: just run the bash script, which uses
`papermill` to execute this notebook non-interactively:

```bash
bash scripts/host/02_compile.sh yolov5 yolov5n \
     data/weights/yolov5n_lpr.pt \
     data/calib/
```

**B — Interactive**: launch JupyterLab inside the container yourself:

```bash
docker run --rm -it --gpus all \
    -v $(pwd):/workspace -w /workspace \
    -p 8888:8888 \
    xilinx/vitis-ai-pytorch-gpu:3.5.0.001 \
    bash -lc 'source /opt/vitis_ai/conda/etc/profile.d/conda.sh && \
             conda activate vitis-ai-pytorch && \
             jupyter lab --ip=0.0.0.0 --no-browser --allow-root'
```

Then open this notebook in your browser at `http://localhost:8888`.

## Configuration

Edit the next cell to choose family + variant. Restart the kernel after
changes — the lpr_pipeline package is imported at startup.

## 1. Configuration

The four parameters that vary per compile job. **Only edit this cell.**

In [ ]:
# ────────────────────────────────────────────────────────────────────
# Edit these four lines, then run all cells in order.
# ────────────────────────────────────────────────────────────────────
FAMILY      = "yolov5"      # yolov5 | yolox | (yolov7/yolov4_csp/ssd_mobilenetv2 = stub)
VARIANT     = "yolov5n"     # see lpr_pipeline/shared/models.py for full list
WEIGHTS     = "data/weights/yolov5n_lpr.pt"
CALIB_DIR   = "data/calib"

# Less commonly changed:
NUM_CLASSES = 1             # number of classes in your trained model
N_CALIB     = 200           # number of calib images to sample
OUTPUT      = None          # None → out/<variant>/<variant>_kv260.xmodel

# Verify the variant exists in the registry
import sys
sys.path.insert(0, "/workspace")
from lpr_pipeline.shared.models import get_spec, list_full_support
spec = get_spec(VARIANT)
print(f"Variant: {VARIANT}")
print(f"  family : {spec.family}")
print(f"  imgsz  : {spec.imgsz}")
print(f"  status : {spec.status}")
print(f"  notes  : {spec.notes}")
if spec.status != "full":
    print(f"\n  ⚠ This variant is a STUB. Compile will raise NotImplementedFamilyError.")
    print(f"  Fully-supported variants: {list_full_support()}")

## 2. Container sanity check

Verifies we're inside the Vitis-AI Docker. If you see ImportErrors, you're
running outside the container — see "Prerequisites" above.

In [ ]:
import importlib, sys

required = {
    "torch":           "PyTorch — base framework",
    "pytorch_nndct":   "vai_q_pytorch — Vitis-AI quantizer (the Docker container provides this)",
}

ok = True
for mod, desc in required.items():
    try:
        m = importlib.import_module(mod)
        v = getattr(m, "__version__", "?")
        print(f"  ✓ {mod:20s} {v:15s}  ({desc})")
    except ImportError as e:
        ok = False
        print(f"  ✗ {mod:20s} MISSING        ({desc})")

import shutil
if shutil.which("vai_c_xir"):
    print(f"  ✓ {'vai_c_xir':20s} on PATH")
else:
    ok = False
    print(f"  ✗ {'vai_c_xir':20s} NOT on PATH")

if not ok:
    print()
    print("  ┌─ Run the compile via the script instead ─────────────")
    print("  │  bash scripts/host/02_compile.sh \\")
    print(f"  │       {FAMILY} {VARIANT} {WEIGHTS} {CALIB_DIR}/")
    print("  └─────────────────────────────────────────────────────")

## 3. Input validation

Confirms the weights file exists and the calibration directory has enough
images. We don't actually load anything yet — fast fail before the heavy
quantization step.

In [ ]:
from pathlib import Path

WEIGHTS_P   = Path("/workspace") / WEIGHTS
CALIB_P     = Path("/workspace") / CALIB_DIR
OUT_P       = Path("/workspace") / (OUTPUT or f"out/{VARIANT}/{VARIANT}_kv260.xmodel")
WORK_P      = Path("/workspace/build") / VARIANT

print(f"weights   : {WEIGHTS_P}")
print(f"  exists  : {WEIGHTS_P.is_file()}")
if WEIGHTS_P.is_file():
    print(f"  size    : {WEIGHTS_P.stat().st_size/1e6:.1f} MB")
print(f"calib_dir : {CALIB_P}")
n_calib = sum(1 for p in CALIB_P.iterdir()
              if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")) \
          if CALIB_P.is_dir() else 0
print(f"  images  : {n_calib}")
if n_calib < 50:
    print(f"  ⚠ {n_calib} images is too few — aim for ≥200 for good quantization accuracy")
print(f"output    : {OUT_P}")
print(f"work_dir  : {WORK_P}")

assert WEIGHTS_P.is_file(),    f"weights not found at {WEIGHTS_P}"
assert CALIB_P.is_dir(),       f"calib dir not found at {CALIB_P}"
assert n_calib >= 1,           f"no calibration images found in {CALIB_P}"

## 4. Run compile

Calls into `lpr_pipeline.compile.<FAMILY>.Compiler.run()`. Heavy lifting —
expect 5-10 minutes for YOLOv5n with 200 calibration images on an RTX 3060.

In [ ]:
from lpr_pipeline.compile import get_compiler
from lpr_pipeline.compile.base import CompileInputs, CompileError, NotImplementedFamilyError

inputs = CompileInputs(
    spec       = spec,
    weights    = WEIGHTS_P,
    calib_dir  = CALIB_P,
    work_dir   = WORK_P,
    out_xmodel = OUT_P,
    nc         = NUM_CLASSES,
    n_calib    = N_CALIB,
)

try:
    compiler   = get_compiler(FAMILY)
    final_path = compiler.run(inputs)
    print(f"\n══ DONE ══")
    print(f"Wrote: {final_path}")

except NotImplementedFamilyError as e:
    print(f"\n[stub] {e}")

except CompileError as e:
    print(f"\n[error] {e}")
    raise

## 5. Post-compile sanity

If we got here, the xmodel exists. A couple of quick reads to confirm
it's compiled for KV260 B4096 (fingerprint `0x101000056010407`).

In [ ]:
if OUT_P.is_file():
    print(f"  ✓ {OUT_P}  ({OUT_P.stat().st_size/1e6:.2f} MB)")
    # Quick sanity via xir
    try:
        import xir
        g = xir.Graph.deserialize(str(OUT_P))
        sgs = g.get_root_subgraph().toposort_child_subgraph()
        dpu_sgs = [s for s in sgs if s.has_attr("device") and s.get_attr("device") == "DPU"]
        cpu_sgs = [s for s in sgs if s.has_attr("device") and s.get_attr("device") == "CPU"]
        print(f"  subgraphs: {len(dpu_sgs)} DPU + {len(cpu_sgs)} CPU = {len(sgs)} total")
        for s in dpu_sgs:
            if s.has_attr("dpu_fingerprint"):
                fp = s.get_attr("dpu_fingerprint")
                print(f"  fingerprint: 0x{fp:016x}")
                if fp == 0x101000056010407:
                    print("    ✓ KV260 B4096 / VAI 3.5")
                else:
                    print("    ⚠ unexpected fingerprint")
                break
    except ImportError:
        print("  (xir not available in this kernel — skip fingerprint check)")
else:
    print(f"  ✗ {OUT_P} not found")

## 6. Next steps

The xmodel is ready to deploy. From the host PC (outside this container):

```bash
bash scripts/host/03_sync_to_kria.sh ubuntu@<board-ip> <variant>
```

(replace `<variant>` with the value of `VARIANT` from cell 1)

That copies the xmodel to the Kria and verifies its sha256 on arrival.
Then on the Kria, run the live demo (notebook `02_deploy_live.ipynb` —
delivered in Pass 6) or batch evaluation (notebook `03_deploy_eval.ipynb`).

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `vai_q_pytorch did not produce an .xmodel` | Calibration failed silently — usually OOM or shape mismatch | Lower `N_CALIB`; check the output of cell 4 for the actual error |
| `Could not find a Detect head in the loaded checkpoint` | Wrong model architecture | Verify your `.pt` is from the matching family — Ultralytics for yolov5, Megvii for yolox |
| `vai_c_xir failed: invalid fingerprint` | Wrong arch.json — sometimes happens if VAI Docker tag drifted | Use the pinned `xilinx/vitis-ai-pytorch-gpu:3.5.0.001` tag |
| Compile crashes in calib step with NaN | Image preprocessing mismatch (e.g. /255 missing) | Inspect `lpr_pipeline/compile/<family>.py` calib loader; should match training |
| OOM on the GPU | Too many calib images at once | Lower `N_CALIB`. The pipeline already uses batch_size=1 |